# unit03 レッスン: はじめての回帰モデル(scikit-learn)

**このレッスンで作れるようになるもの**: 442人分の医療データから「1年後の病状の進行度」を予測するモデルを、**わずか数行**で学習させ、その予測がどれくらい当たっているかを数値で評価する。

これが機械学習の**最小の一周**です。データを分ける → モデルに学習させる(`fit`)→ 予測させる(`predict`)→ 評価する。この4ステップは、この先どんなモデルを使っても**まったく同じ形**で登場します。

- 所要時間: 15〜25分
- 前提: unit01(NumPy)・unit02(pandas)は済んでいること。**機械学習と scikit-learn は初めてでOK**
- 進め方: セルを上から順に実行(`Shift+Enter`)。「書いてみる」セルだけ自分で書く
- 詰まったら: Claude に聞いてOK(答えではなくヒントをくれます)

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def check(name, actual, expected, hint=""):
    try:
        ok = actual is not None and bool(np.all(np.isclose(np.asarray(actual, dtype=float), np.asarray(expected, dtype=float), rtol=1e-2)))
    except (TypeError, ValueError):
        ok = actual == expected
    if ok:
        print(f"[OK] {name}: 正解!")
    else:
        print(f"[NG] {name}: 期待値 {expected!r} / 実際 {actual!r}")
        if hint:
            print(f"     ヒント: {hint}")
    return ok

print("準備OK! これから使うデータを見てみます。")
X, y = load_diabetes(return_X_y=True)
print("特徴量 X の形:", X.shape, " (442人 × 10項目の検査値)")
print("目的変数 y の形:", y.shape, " (442人ぶんの、1年後の病状進行度)")

---
## 概念1: モデルの「学習(fit)」と「予測(predict)」 — estimator API

### なぜ学ぶか
「過去のデータからパターンを学んで、未知のデータを予測する」— これが機械学習の仕事です。中古車の価格査定、明日の来客数の予測、迷惑メールの判定。実務では**まず既存データでモデルを学習させ、それを本番の新しいデータに当てて予測を出す**という流れになります。scikit-learn はこの「学習して予測する」部品を大量に用意していて、求人票の「scikit-learn 経験」はこの使い方を指します。

### 解説

機械学習の「モデル」= 入力(特徴量 X)から出力(目的変数 y)を予測する箱です。この箱に対してやることは、たった2つ:

- **`fit(X, y)`(学習)**: 「Xとyの正解ペア」を大量に見せて、箱の中のパラメータを調整する。**これが"学習"の正体**です。C# で言えば、オブジェクトのフィールドを書き換える副作用メソッド。戻り値ではなく、**呼んだあとのモデル自身が学習済みになる**。
- **`predict(X)`(予測)**: 学習済みの箱に新しい X を入れて、y の予測値を出させる。こちらはモデルを変更しません(何度呼んでも同じ)。

scikit-learn の強力な点は、**どのモデルもこの同じ `fit` / `predict` を持つ**こと。線形回帰でも決定木でもニューラルネットでも、使う側の手順は変わりません。

```
C#:      interface IEstimator { void Fit(X, y); Y Predict(X); }
         // LinearRegression も DecisionTree も同じインターフェースを実装
scikit-learn:  model.fit(X, y)  →  model.predict(X)   # モデルを差し替えても呼び方は同じ
```

この「共通の呼び方」を **estimator API** と呼びます。今回使う **`LinearRegression`(線形回帰)** は、その estimator の一種です。まず動くコードを見ます。次のセルを実行して、出力を観察してください。

In [ ]:
# GOAL: モデルを1個作って fit→predict する、機械学習の最小の一周を見る

# STEP 1: モデルのインスタンスを生成 — この時点では「空箱」でまだ何も学習していない
#         (C#: new LinearRegression() と同じ。まだ Fit を呼んでいない状態)
model = LinearRegression()

# STEP 2: fit で学習 — X(10項目)と y(進行度)の対応を箱に覚えさせる
#         fit は戻り値ではなく model 自身を書き換える(だから model = ... とは書かない)
model.fit(X, y)
print("学習完了。モデルは学習済み状態になった。")

# STEP 3: predict で予測 — 学習に使ったのと同じXを入れて、予測値を出させてみる
preds = model.predict(X)
print("最初の3人の予測値:", np.round(preds[:3], 1))
print("最初の3人の正解値:", y[:3])
print("→ 完璧ではないが、だいたい近い値を出せている")

### 予測してみよう

次のセルでは、同じ `model` に対して **`predict` を2回**呼びます。

**実行する前に**予測してください: 2回の予測結果は同じになるでしょうか、違うでしょうか?(ヒント: `predict` はモデルを書き換えるか?)

In [ ]:
# 予測してから実行!
p1 = model.predict(X)
p2 = model.predict(X)
print("2回の予測は完全に一致する?:", np.array_equal(p1, p2))

予測は当たりましたか? `predict` はモデルを**読むだけ**で書き換えないので、何度呼んでも同じ結果です。「学習は `fit`、予測は `predict` で何度でも」と覚えてください。

### 書いてみる

**課題**: 新しく `LinearRegression()` を作って `X`, `y` で `fit` し、`predict(X)` で予測値の配列を出してください。その予測値の**個数**(`len(...)`)を `result1` に入れてください。

ヒント(概念レベル): 予測値は「1人につき1つ」出る。データは442人ぶん。

In [ ]:
result1 = None
# ここに書く(新しいモデルを fit → predict し、予測値の個数を result1 に代入)


check("概念1: fit/predict の一周", result1, 442,
      hint="predict(X) は入力した人数ぶんの予測値を返す。X.shape[0] と同じ個数になるはず")

---
## 概念2: データを訓練用とテスト用に分ける — なぜ分けるのか

### なぜ学ぶか
概念1では**学習に使ったデータで予測して**「近い値が出た」と喜びました。でもこれは**カンニング**です。実務でモデルを本番投入する前に必ず問われるのは「**まだ見たことのないデータ**でどれくらい当たるのか?」。これに答えられないモデルは誰も信用しません。中古車査定モデルを納品する時、顧客は「学習に使った車で当たった」ではなく「新しく持ち込まれた車で当たる」ことを知りたいのです。

### 解説

学習に使ったデータで評価すると、点数が**不当に良く出ます**。モデルは答えを丸暗記できてしまうからです(試験前に答えを見てから同じ試験を受けるようなもの)。

そこで、データを最初に2つに分けます:

- **訓練データ(train)**: モデルの学習(`fit`)に使う。
- **テストデータ(test)**: 学習には**絶対に使わず**、最後の評価だけに使う。「本番の未知データ」の代役。

この分割をやってくれるのが **`train_test_split`(データを訓練用とテスト用に2分割する関数)**です:

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
```

- `test_size=0.2` → 全体の **20%をテスト**に、残り80%を訓練に回す。
- `random_state=42` → 分け方はランダムだが、**この数を固定すると毎回同じ分割**になる(再現性のため。数字自体に意味はなく、42でも0でも何でもよい。チーム全員で揃えるのが目的)。

返り値は**必ずこの4つの順番**(train2つ → test2つ)。ここを取り違えるとバグります。次のセルで確認します。

In [ ]:
# GOAL: train_test_split が「4つ」を決まった順で返すこと、分割の件数を確認する

# STEP 1: 20%をテストに回して4分割 — 返り値の順番は (X_train, X_test, y_train, y_test) で固定
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# STEP 2: 件数を確認 — 442人の20%がテスト、残りが訓練
print("全体      :", X.shape[0], "人")
print("訓練(train):", X_train.shape[0], "人  <- fit に使う")
print("テスト(test):", X_test.shape[0], "人  <- 評価だけに使う(学習には使わない)")

# STEP 3: 訓練データ「だけ」で学習し、テストデータで予測(=本番のシミュレーション)
model = LinearRegression().fit(X_train, y_train)   # fit は自分自身を返すので繋げて書ける
preds = model.predict(X_test)
print("テスト1人目の 予測:", round(float(preds[0]), 1), " / 正解:", y_test[0])

### 予測してみよう

次のセルは `test_size=0.3`(テストを30%に増やす)で分割し直します。

**実行する前に**予測してください: **訓練データ(train)の件数**は、さっき(test_size=0.2 のとき 353人)から**増えますか、減りますか?** だいたい何人になりそうですか?(全体442人)

In [ ]:
# 予測してから実行!
Xtr3, Xte3, ytr3, yte3 = train_test_split(X, y, test_size=0.3, random_state=42)
print("test_size=0.3 のとき  訓練:", Xtr3.shape[0], "人 / テスト:", Xte3.shape[0], "人")

予測は当たりましたか? テストを増やせば、その分**訓練は減ります**(442 × 0.7 ≒ 309人)。テストが多いほど評価は安定しますが、訓練が減ると学習が弱くなる — このトレードオフがあるので実務では 0.2〜0.3 がよく使われます。

### 書いてみる

**課題**: `X`, `y` を `test_size=0.2, random_state=42` で分割し、**訓練データ(X_train)の件数**を `result2` に入れてください。

ヒント(概念レベル): 返り値は4つで順番は (X_train, X_test, y_train, y_test)。件数は `.shape[0]`。

In [ ]:
result2 = None
# ここに書く(split して X_train の件数を result2 に代入)


check("概念2: train_test_split", result2, 353,
      hint="test_size=0.2 は「テストが20%」の意味。訓練は残りの80%。442×0.8 は?")

---
## 概念3: 予測の良し悪しを数値で測る — 回帰の指標(MAE / R2)

### なぜ学ぶか
「モデルができました」だけでは実務は動きません。上司や顧客は必ず「**で、どれくらい当たるの?**」と聞きます。分類(迷惑メールか否か)なら「正解率90%」で一言ですが、今回のような**数値を当てる問題(回帰)**は「ピタリ正解」がほぼないので、「**平均でどれくらいズレたか**」を測る指標が必要です。指標を読めることは、モデルを比較して「どっちが良いか」を判断する土台になります。

### 解説

回帰の代表的な指標は2つ。どちらも `y_test`(正解)と `preds`(予測)を渡すだけ:

- **MAE(平均絶対誤差)** = 予測が平均で**何ズレたか**。単位は y と同じで直感的。
  `mean_absolute_error(y_test, preds)`。MAEが43なら「平均43ずれる」— 小さいほど良い。
- **R2(決定係数)** = モデルが y のばらつきを**どれだけ説明できたか**。`r2_score(y_test, preds)`。
  - **1.0** = 完璧、**0.0** = 「常に平均値を答えるだけのモデル」と同レベル(=学習した意味なし)、**マイナス** = それ以下(むしろ悪い)。
  - 「割合」なので単位に依らず、**異なるデータ間でもモデルの良さを比べやすい**。

覚え方: **MAE は「ズレの大きさ(小さいほど良い、単位つき)」、R2 は「説明できた割合(1に近いほど良い、0〜1が目安)」**。次のセルで両方を出します。

In [ ]:
# GOAL: 訓練→予測→評価まで通し、MAE と R2 の値を目で確認する

# STEP 1: 訓練データで学習し、テストデータで予測(概念2の続き)
model = LinearRegression().fit(X_train, y_train)
preds = model.predict(X_test)

# STEP 2: MAE — 予測が平均で何ズレたか(単位は y と同じ「進行度」)
mae = mean_absolute_error(y_test, preds)
print("MAE:", round(mae, 2), " -> 予測は平均でこれくらいズレる(小さいほど良い)")

# STEP 3: R2 — ばらつきをどれだけ説明できたか(1が完璧、0が平均を答えるだけ)
r2 = r2_score(y_test, preds)
print("R2 :", round(r2, 3), " -> 0.45なら『ばらつきの45%を説明できた』の意味")

### 予測してみよう

「常に**訓練データの平均値**を予測するだけ」の、何も学習しないダメなモデルを考えます。次のセルでその MAE と R2 を出します。

**実行する前に**予測してください: この「平均を答えるだけモデル」の **R2 はいくつに近くなる**でしょう?(解説の R2 の説明を思い出して)

In [ ]:
# 予測してから実行!
dummy_preds = np.full_like(y_test, y_train.mean(), dtype=float)  # 全員に「訓練の平均」を予測
print("平均予測モデル MAE:", round(mean_absolute_error(y_test, dummy_preds), 2))
print("平均予測モデル R2 :", round(r2_score(y_test, dummy_preds), 3))

予測は当たりましたか? 「平均を答えるだけ」の R2 は **ほぼ0**(このテストデータでは約 -0.001)。だから R2=0 は「学習した意味がゼロ」の基準線で、さっきの本物のモデル(R2≒0.45)は**確かに何かを学べている**と分かります。MAE も本物の方が小さいはずです。

### 書いてみる

**課題**: 上で作った学習済み `model` の予測 `preds` について、テストデータでの **MAE** を計算し `result3` に入れてください(`model` と `preds`・`y_test` は上のセルで作成済み。作り直してもOK)。

ヒント(概念レベル): `mean_absolute_error(正解, 予測)` に `y_test` と `preds` を渡すだけ。

In [ ]:
result3 = None
# ここに書く(y_test と preds から MAE を計算して result3 に代入)


check("概念3: 回帰指標(MAE)", result3, 42.79,
      hint="mean_absolute_error(y_test, preds)。引数の順番は (正解, 予測)。値は40台になるはず")

---
## 振り返り(1〜2文でOK — このセルを編集して書き込んでください)

- **今日学んだことを自分の言葉で**:
- **難しかったこと(あれば)**:

(この記述はセッション終了時にチューターが学習ノートとスキルレベル判定に使います)

## まとめと次へ

| 概念 | 一言で | C#で言うと |
|------|--------|-----------|
| estimator API (fit/predict) | `model.fit(X,y)` で学習、`model.predict(X)` で予測。どのモデルも同じ形 | 全モデルが共通 `IEstimator` を実装 |
| train_test_split | データを訓練/テストに分けてカンニングを防ぐ。`random_state` で再現性 | データを2つに分割する前処理 |
| 回帰指標 (MAE / R2) | MAE=平均のズレ(小さいほど良い)、R2=説明できた割合(1に近いほど良い) | 「距離」を測る複数の Comparer |

**この4ステップ(分割 → fit → predict → 評価)がこの先ずっと基本形です。**

**この先どこで使うか**: **unit04(分類)では、`LinearRegression` を `LogisticRegression` に差し替えるだけで、`fit`/`predict`/`train_test_split` は今日とまったく同じコードが動きます**(estimator API のおかげ)。変わるのは指標だけ(回帰の MAE/R2 → 分類の正解率など)。unit05 の総合プロジェクトでは、この一周に前処理(unit02)を足して一気通貫のパイプラインを書きます。

**次**: 演習 `ex01_split_fit_predict.py` へ。lesson を見ながらで OK。テストは
`python -m pytest courses/ml-intro/unit03-first-model/tests/test_ex01.py -q`